# T-302 / T-303 — Verifier fine-tune (SciFact → DeBERTa-v3)

**ResearchGPT**, Phase 3. This notebook trains **the graded component**: the
three-way claim verifier that decides whether a drafted claim is `SUPPORT`,
`CONTRADICT` or `NEI` against a retrieved evidence sentence.

| | |
|---|---|
| Task | T-302 (author) / T-303 (run) |
| Base | `cross-encoder/nli-deberta-v3-small` (142M total: ~44M backbone + a 128k-token embedding matrix; ungated, safetensors) |
| Data | `allenai/scifact_entailment`, paired per TASKLIST T-301 |
| Runtime | **GPU T4 x2** (single GPU is used; ~10–20 min) |
| Internet | **must be ON** (Settings → Internet) |
| Output | `verifier-scifact/` checkpoint + `verifier_metrics.json` |

### What it produces
1. A fine-tuned checkpoint, ready for the T-304 ONNX int8 export.
2. **The T-305 table** — stock vs fine-tuned on the *same* untouched held-out
   set, which is the headline result justifying the training component.
3. **The T-307 verdict** — escalate to `deberta-v3-base` or not, against the
   0.75 macro-F1 threshold fixed in `decisions.txt` D-09 *before* this ran.

### Two things that are easy to get wrong, and are handled here
* **Pair order is `(evidence, claim)`** — premise then hypothesis. This is an
  NLI model; reversing them silently trains a different task.
* **The stock head is `{0: contradiction, 1: entailment, 2: neutral}`**, which
  is *not* our label order. The head is **permuted, not reinitialised**, so the
  fine-tune starts from a model that already knows entailment. The same
  permutation is what lets the stock arm be scored fairly in T-305.

## 1 · Configuration

`BASE_MODEL` is the only line to change for the **T-307 escalation**: swap in
`cross-encoder/nli-deberta-v3-base` and re-run the whole notebook top to bottom.
Everything else is deliberately fixed — `SEED` in particular, because the T-301
pairing is a pure function of it and the local `data/scifact/*.jsonl` must match
what trains here.

In [ ]:
# --- T-307 escalation: change ONLY this line and re-run -----------------
BASE_MODEL = "cross-encoder/nli-deberta-v3-small"     # 142M total (~44M backbone)
# BASE_MODEL = "cross-encoder/nli-deberta-v3-base"    # ~184M backbone, escalation target
# -----------------------------------------------------------------------

SEED            = 20260827   # must equal researchgpt.verify.scifact_pairs.SEED
NEI_PER_CLAIM   = 2          # must equal the local build
DEV_FRAC        = 0.15
EPOCHS          = 3          # TASKLIST T-302: 2-3
LEARNING_RATE   = 2e-5
BATCH_SIZE      = 32
MAX_LENGTH      = 256
OUTPUT_DIR      = "/kaggle/working/verifier-scifact"

ESCALATION_THRESHOLD = 0.75  # decisions.txt D-09, fixed before this ran
LABELS = ("SUPPORT", "CONTRADICT", "NEI")
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

print("base model :", BASE_MODEL)
print("labels     :", LABELS)

## 2 · Dependencies

Kaggle's images ship `transformers`, `torch`, `sentencepiece` and `protobuf`
already — the last two matter because **DeBERTa-v3 uses a SentencePiece
tokenizer** and fails with a misleading "conversion" error without them
(TASKLIST T-306 records them missing on the local machine; Kaggle has them).

The cell is tolerant: it installs only what is actually absent, so a re-run
costs seconds rather than re-resolving the whole environment.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg.split("==")[0].replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, mod in [("transformers", "transformers"),
                 ("accelerate>=0.26", "accelerate"),   # Trainer hard-requires it
                 ("sentencepiece", "sentencepiece"),
                 ("protobuf", "google.protobuf"),
                 ("pyarrow", "pyarrow"),
                 ("huggingface_hub", "huggingface_hub"),
                 ("scikit-learn", "sklearn")]:
    ensure(pkg, mod)

import torch, transformers
print("\ntorch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("cuda         :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3 · Build the T-301 pairs

⚠️ **`allenai/scifact_entailment` cannot be loaded with `load_dataset`.** It is a
*script-based* dataset and `datasets>=4` refuses it outright
(`RuntimeError: Dataset scripts are no longer supported`). TASKLIST T-301 says
this is true of plain `allenai/scifact` only — verified 2026-08-27, it is true
of `scifact_entailment` as well.

The fix is the Hub's auto-converted **parquet branch**, `refs/convert/parquet`,
which carries identical rows with no script and needs no token.

> **This cell is a mirror of `src/researchgpt/verify/scifact_pairs.py`.** Keep
> the two in sync — `tests/test_phase3_acceptance.py` asserts that this notebook
> and the module produce byte-identical pairs, and fails if they drift.

In [ ]:
import json, random
from collections import Counter
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

_REPO, _REVISION = "allenai/scifact_entailment", "refs/convert/parquet"
EXPECTED_ROWS = {"train": 919, "validation": 340}

def load_split(split):
    path = hf_hub_download(_REPO, f"default/{split}/0000.parquet",
                           repo_type="dataset", revision=_REVISION)
    rows = pq.read_table(path).to_pylist()
    exp = EXPECTED_ROWS.get(split)
    if exp is not None and len(rows) != exp:
        print(f"  WARNING: {split}: got {len(rows)} rows, expected {exp}.")
    return rows

def build_pairs(rows, *, nei_per_claim=2, seed=SEED):
    # SUPPORT/CONTRADICT -> one pair per annotated evidence sentence.
    # NEI -> sample `nei_per_claim` sentences from that row's abstract.
    rng, pairs = random.Random(seed), []
    for row in rows:
        verdict = row["verdict"]
        sentences = list(row["abstract"])
        if not sentences:
            continue
        if verdict == "NEI":
            k = min(nei_per_claim, len(sentences))
            indices = sorted(rng.sample(range(len(sentences)), k))
        else:
            indices = [i for i in row["evidence"] if 0 <= i < len(sentences)]
            if not indices:
                continue
        for i in indices:
            pairs.append({"claim": row["claim"], "evidence": sentences[i],
                          "label": verdict, "claim_id": row["claim_id"],
                          "abstract_id": row["abstract_id"], "sentence_index": i})
    return pairs

def split_by_claim(pairs, *, dev_frac=DEV_FRAC, seed=SEED):
    # Group by claim_id. A row-level split leaks: one claim spans up to 5
    # abstracts, so the same claim text would sit on both sides.
    claim_ids = sorted({p["claim_id"] for p in pairs})
    rng = random.Random(seed); rng.shuffle(claim_ids)
    dev_ids = set(claim_ids[:int(round(len(claim_ids) * dev_frac))])
    return ([p for p in pairs if p["claim_id"] not in dev_ids],
            [p for p in pairs if p["claim_id"] in dev_ids])

raw_train, raw_val = load_split("train"), load_split("validation")
all_train = build_pairs(raw_train, nei_per_claim=NEI_PER_CLAIM)
val_pairs = build_pairs(raw_val,  nei_per_claim=NEI_PER_CLAIM)
train_pairs, dev_pairs = split_by_claim(all_train)

assert not ({p["claim_id"] for p in train_pairs} & {p["claim_id"] for p in dev_pairs}), \
    "claim_id leaked across the train/dev split"

for name, ps in [("train", train_pairs), ("dev", dev_pairs), ("validation", val_pairs)]:
    c = Counter(p["label"] for p in ps)
    print(f"  {name:12s} pairs {len(ps):5d}  claims {len({p['claim_id'] for p in ps}):4d}  "
          + "  ".join(f"{k} {c.get(k,0):4d}" for k in LABELS))

print("\n  OK: no claim_id appears in both train and dev")

## 4 · Tokenize

**Premise = evidence, hypothesis = claim.** One helper, used by training and by
every evaluation below, so the order cannot drift between them.

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("tokenizer:", type(tokenizer).__name__)

def encode_pair(pair):
    # (premise, hypothesis) = (evidence sentence, claim under test).
    return pair["evidence"], pair["claim"]

class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=MAX_LENGTH):
        premises   = [encode_pair(p)[0] for p in pairs]
        hypotheses = [encode_pair(p)[1] for p in pairs]
        self.enc = tokenizer(premises, hypotheses, truncation=True,
                             max_length=max_length, padding=False)
        self.labels = [LABEL2ID[p["label"]] for p in pairs]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds = PairDataset(train_pairs, tokenizer)
dev_ds   = PairDataset(dev_pairs,   tokenizer)
val_ds   = PairDataset(val_pairs,   tokenizer)

lens = [len(x) for x in train_ds.enc["input_ids"]]
print(f"train {len(train_ds)}  dev {len(dev_ds)}  validation {len(val_ds)}")
print(f"token length: mean {sum(lens)/len(lens):.0f}  max {max(lens)}  "
      f"truncated at {MAX_LENGTH}: {sum(1 for l in lens if l >= MAX_LENGTH)}")

## 5 · Warm start — permute the head, do not reinitialise it

`cross-encoder/nli-deberta-v3-*` already has a trained 3-way NLI head, but in
the order `{0: contradiction, 1: entailment, 2: neutral}`. Ours is
`(SUPPORT, CONTRADICT, NEI)` = `(entailment, contradiction, neutral)`.

Loading with `ignore_mismatched_sizes` or a fresh head throws away a head that
already solves 2/3 of this problem. **Permuting rows `[1, 0, 2]` keeps it.**
The permutation is derived from the checkpoint's own `config.id2label` and
asserted, never hard-coded.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoConfig

NLI_TO_VERDICT = {"entailment": "SUPPORT",
                  "contradiction": "CONTRADICT",
                  "neutral": "NEI"}

def label_permutation(id2label):
    # Column indices reordering a checkpoint's logits into LABELS order.
    normalised = {}
    for idx, raw in id2label.items():
        name = str(raw).strip()
        verdict = name.upper() if name.upper() in LABEL2ID else NLI_TO_VERDICT.get(name.lower())
        if verdict is None:
            raise ValueError(f"cannot map checkpoint label {raw!r} onto {LABELS}")
        normalised[verdict] = int(idx)
    missing = [l for l in LABELS if l not in normalised]
    if missing:
        raise ValueError(f"checkpoint head missing {missing}; id2label={id2label}")
    return [normalised[l] for l in LABELS]

stock_config = AutoConfig.from_pretrained(BASE_MODEL)
PERM = label_permutation(stock_config.id2label)
print("stock id2label :", stock_config.id2label)
print("permutation    :", PERM, "->", LABELS)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

with torch.no_grad():
    w, b = model.classifier.weight.data, model.classifier.bias.data
    model.classifier.weight.data = w[PERM].clone()
    model.classifier.bias.data   = b[PERM].clone()

print("head permuted into", LABELS, "order")
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 6 · Metrics

Hand-written rather than pulled from sklearn so the zero-division convention is
explicit: a class the model never predicts scores precision **0**, not NaN. This
mirrors `researchgpt.verify.evaluate.score` exactly.

In [ ]:
import numpy as np

def score(y_true, y_pred):
    n = len(LABELS)
    confusion = [[0]*n for _ in range(n)]
    for t, p in zip(y_true, y_pred):
        confusion[int(t)][int(p)] += 1
    per_class, f1s = {}, []
    for i, label in enumerate(LABELS):
        tp = confusion[i][i]
        fp = sum(confusion[j][i] for j in range(n)) - tp
        fn = sum(confusion[i]) - tp
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        per_class[label] = {"precision": prec, "recall": rec, "f1": f1,
                            "support": sum(confusion[i])}
        f1s.append(f1)
    return {"per_class": per_class, "macro_f1": sum(f1s)/n,
            "accuracy": sum(confusion[i][i] for i in range(n))/max(len(y_true), 1),
            "confusion": confusion, "n": len(y_true)}

def report(name, r):
    print(f"{name}   n={r['n']}  accuracy={r['accuracy']:.3f}  macro-F1={r['macro_f1']:.3f}")
    print(f"  {'class':12s} {'P':>7s} {'R':>7s} {'F1':>7s} {'support':>8s}")
    for label, m in r["per_class"].items():
        print(f"  {label:12s} {m['precision']:7.3f} {m['recall']:7.3f} "
              f"{m['f1']:7.3f} {m['support']:8d}")
    print(f"  confusion (rows=true {list(LABELS)}, cols=pred):")
    for i, row in enumerate(r["confusion"]):
        print(f"    {LABELS[i]:12s} {row}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).argmax(-1)
    r = score(list(labels), list(preds))
    out = {"accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
    for label, m in r["per_class"].items():
        out[f"f1_{label}"] = m["f1"]
    return out

## 7 · Stock baseline — **before** training

Scored here, on the untouched `validation` split, through the same permutation.
This is the `stock` row of the T-305 table. Taking it now means the comparison
is against the identical data and identical code path as the tuned arm.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer)
device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def predict(m, dataset, permutation=None, batch_size=64):
    m.eval().to(device)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator)
    preds = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("labels", None)
        logits = m(**batch).logits
        if permutation is not None:
            logits = logits[:, permutation]
        preds.extend(logits.argmax(-1).cpu().tolist())
    return preds

stock_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
y_true = [LABEL2ID[p["label"]] for p in val_pairs]
stock_result = score(y_true, predict(stock_model, val_ds, permutation=PERM))
report("stock (zero-shot)", stock_result)

del stock_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8 · Train

Cross-entropy, fixed seed, `macro_f1` on the claim-grouped **dev** split chooses
the best checkpoint. The held-out `validation` split is not looked at until
section 9 — it is the number that gets reported, so it must not steer training.

In [ ]:
from transformers import Trainer, TrainingArguments, set_seed

set_seed(SEED)

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\ntrain runtime: %.1f s" % train_result.metrics["train_runtime"])

## 9 · T-305 — stock vs fine-tuned on the held-out set

The first time `validation` is used for anything.

In [ ]:
tuned_result = score(y_true, predict(trainer.model, val_ds))
report("fine-tuned", tuned_result)

print("\n" + "="*72)
print("T-305 - STOCK vs FINE-TUNED (same held-out set, n=%d)" % tuned_result["n"])
print("="*72)
hdr = f"  {'arm':14s} {'acc':>7s} {'macro-F1':>9s} " + " ".join(f"{l[:9]+' F1':>13s}" for l in LABELS)
print(hdr)
for name, r in (("stock", stock_result), ("fine-tuned", tuned_result)):
    row = f"  {name:14s} {r['accuracy']:7.3f} {r['macro_f1']:9.3f} "
    row += " ".join(f"{r['per_class'][l]['f1']:13.3f}" for l in LABELS)
    print(row)
print(f"  {'delta':14s} {tuned_result['accuracy']-stock_result['accuracy']:+7.3f} "
      f"{tuned_result['macro_f1']-stock_result['macro_f1']:+9.3f}")

## 10 · T-307 escalation gate

Threshold fixed at **0.75 macro-F1** in `decisions.txt` D-09, *before* this run.
It is applied mechanically so the decision cannot be rationalised after seeing
the number.

In [ ]:
macro_f1 = tuned_result["macro_f1"]
escalate = macro_f1 < ESCALATION_THRESHOLD

print("="*72)
if escalate:
    print(f"T-307 GATE: ESCALATE.  macro-F1 {macro_f1:.3f} < {ESCALATION_THRESHOLD}")
    print("  -> set BASE_MODEL = 'cross-encoder/nli-deberta-v3-base' in section 1")
    print("     and re-run this notebook top to bottom.")
    print("  -> latency is not a constraint (TASKLIST 0.2); cost is Kaggle time only.")
else:
    print(f"T-307 GATE: PASS.  macro-F1 {macro_f1:.3f} >= {ESCALATION_THRESHOLD}")
    print("  -> keep deberta-v3-small.")
    print("  -> spend the remaining Kaggle quota on T-332 (reranker fine-tune).")
print("="*72)

## 11 · Save the checkpoint and metrics

Writes to `/kaggle/working/`, which is what the **Output** tab exposes for
download. Grab **both**: the checkpoint feeds T-304 (ONNX int8 export) and the
metrics JSON feeds T-305's table in the report.

In [ ]:
import os, shutil, json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metrics = {
    "task": "T-302/T-303",
    "base_model": BASE_MODEL,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "nei_per_claim": NEI_PER_CLAIM,
    "labels": list(LABELS),
    "counts": {"train": len(train_pairs), "dev": len(dev_pairs),
               "validation": len(val_pairs)},
    "stock": stock_result,
    "tuned": tuned_result,
    "escalation": {"threshold": ESCALATION_THRESHOLD,
                   "macro_f1": macro_f1,
                   "escalate": bool(escalate)},
    "train_runtime_s": train_result.metrics.get("train_runtime"),
}
with open("/kaggle/working/verifier_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

shutil.make_archive("/kaggle/working/verifier-scifact", "zip", OUTPUT_DIR)
shutil.rmtree("/kaggle/working/checkpoints", ignore_errors=True)

print("saved:")
for f in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", f)
    if os.path.isfile(p):
        print(f"  {f:36s} {os.path.getsize(p)/1e6:8.2f} MB")
print("\nDownload verifier-scifact.zip AND verifier_metrics.json from the Output tab.")
print("Then locally:  python -m researchgpt.verify.export_onnx --checkpoint <unzipped dir>")